In [6]:
#c1
!pip install torch torchvision torchaudio
!pip install rdkit pandas numpy scikit-learn
!pip install torchdiffeq
print("Environment successfully installed.")

Environment successfully installed.


In [7]:
#c2
import pandas as pd
import numpy as np
import torch
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.preprocessing import StandardScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")

url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/qm9.csv"
print("Downloading QM9 dataset (this may take a minute)...")
df = pd.read_csv(url)


df = df[['smiles', 'gap']].dropna()
#sample 10,000 for training
df = df.sample(10000, random_state=42).reset_index(drop=True)

#SMILES to Morgan Fingerprints (128-bit for latent compression)
fps = []
valid_idx = []
for i, smiles in enumerate(df['smiles']):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=128)
        fps.append(list(fp))
        valid_idx.append(i)

fps_array = np.array(fps, dtype=np.float32)
gaps_array = df['gap'].iloc[valid_idx].values.astype(np.float32).reshape(-1, 1)

#normalizing the target gaps
scaler = StandardScaler()
gaps_scaled = scaler.fit_transform(gaps_array)

#converting to PyTorch Tensors
X_tensor = torch.tensor(fps_array).to(device)
y_tensor = torch.tensor(gaps_scaled).to(device)

print(f"Successfully processed {X_tensor.shape[0]} valid molecules.")

Using compute device: cuda


Streaming output truncated to the last 5000 lines.
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:03:48] DEPRECATION WARNING: please use MorganGenerator
[12:0

Successfully processed 10000 valid molecules.


In [8]:
#c3
import torch.nn as nn
import torch.optim as optim

class MolAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        #compress 128 bits to 16D Latent Space
        self.encoder = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 16)
        )
        #decoding
        self.decoder = nn.Sequential(
            nn.Linear(16, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.Sigmoid() # Bits are 0 to 1
        )
        #predicting HOMO-LUMO gap directly from the 16D Latent Space
        self.physics_predictor = nn.Sequential(
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        gap_pred = self.physics_predictor(z)
        return z, x_recon, gap_pred

ae_model = MolAutoencoder().to(device)
ae_optimizer = optim.Adam(ae_model.parameters(), lr=0.005)
mse_loss = nn.MSELoss()

print("Training Autoencoder and Physics Surrogate...")
ae_model.train()
for epoch in range(500):
    ae_optimizer.zero_grad()
    z, x_recon, gap_pred = ae_model(X_tensor)

    # Loss = Reconstruction accuracy + Physics prediction accuracy
    loss_recon = mse_loss(x_recon, X_tensor)
    loss_phys = mse_loss(gap_pred, y_tensor)
    loss = loss_recon + loss_phys

    loss.backward()
    ae_optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch} | Total Loss: {loss.item():.4f} (Recon: {loss_recon.item():.4f}, Phys: {loss_phys.item():.4f})")

with torch.no_grad():
    ae_model.eval()
    latent_vectors = ae_model.encoder(X_tensor)

Training Autoencoder and Physics Surrogate...
Epoch 0 | Total Loss: 1.2458 (Recon: 0.2486, Phys: 0.9973)
Epoch 100 | Total Loss: 0.2935 (Recon: 0.1028, Phys: 0.1907)
Epoch 200 | Total Loss: 0.2031 (Recon: 0.0883, Phys: 0.1147)
Epoch 300 | Total Loss: 0.1697 (Recon: 0.0809, Phys: 0.0888)
Epoch 400 | Total Loss: 0.1497 (Recon: 0.0770, Phys: 0.0728)


In [9]:
#c4
from torchdiffeq import odeint

class VectorField(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, latent_dim)
        )

    def forward(self, t, z):
        # Concatenate time scalar t to the latent vector z
        t_vec = torch.ones(z.shape[0], 1, device=z.device) * t
        tz = torch.cat([t_vec, z], dim=1)
        return self.net(tz)

class FlowMatcher(nn.Module):
    def __init__(self):
        super().__init__()
        self.vf = VectorField()

    def forward(self, z0):
        # Integrate from t=0 (noise) to t=1 (structured latent space)
        t_span = torch.tensor([0.0, 1.0], device=z0.device)
        z_t = odeint(self.vf, z0, t_span, method='euler')
        return z_t[-1]

flow_model = FlowMatcher().to(device)
flow_optimizer = optim.Adam(flow_model.parameters(), lr=0.001)

TARGET_GAP = torch.tensor([[-1.0]], device=device)

print("Training Physics-Informed Flow Network...")
flow_model.train()
for epoch in range(300):
    flow_optimizer.zero_grad()

    # sample pure Gaussian noise
    batch_size = 256
    z0 = torch.randn(batch_size, 16, device=device)

    #flow matching
    z_generated = flow_model(z0)

    #predicting physical properties of vectors
    predicted_gaps = ae_model.physics_predictor(z_generated)

    #phys
    loss_physics = mse_loss(predicted_gaps, TARGET_GAP.expand_as(predicted_gaps))


    loss_dist = mse_loss(z_generated.mean(dim=0), latent_vectors.mean(dim=0)) + \
                mse_loss(z_generated.std(dim=0), latent_vectors.std(dim=0))

    total_loss = loss_physics * 5.0 + loss_dist

    total_loss.backward()
    flow_optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch} | Flow Loss: {total_loss.item():.4f} (Physics Penalty: {loss_physics.item():.4f})")

Training Physics-Informed Flow Network...
Epoch 0 | Flow Loss: 8.7685 (Physics Penalty: 1.6150)
Epoch 50 | Flow Loss: 1.0686 (Physics Penalty: 0.0954)
Epoch 100 | Flow Loss: 0.5735 (Physics Penalty: 0.0332)
Epoch 150 | Flow Loss: 0.3465 (Physics Penalty: 0.0220)
Epoch 200 | Flow Loss: 0.2522 (Physics Penalty: 0.0165)
Epoch 250 | Flow Loss: 0.1933 (Physics Penalty: 0.0133)


In [10]:
#c5
from sklearn.metrics import pairwise_distances

flow_model.eval()
ae_model.eval()

with torch.no_grad():
    #generating 5 candidate ideal latent vectors
    z_noise = torch.randn(5, 16, device=device)
    z_ideal = flow_model(z_noise)

    #decoding to ideal fingerprints
    ideal_fps = ae_model.decoder(z_ideal).cpu().numpy()

    #calculating predicted gaps for these ideal vectors
    ideal_gaps = ae_model.physics_predictor(z_ideal).cpu().numpy()
    ideal_gaps_real = scaler.inverse_transform(ideal_gaps)

real_fps = X_tensor.cpu().numpy()
distances = pairwise_distances(ideal_fps, real_fps, metric='manhattan')

print("\n--- TOP GENERATED CANDIDATES ---")
for i in range(5):
    best_idx = np.argmin(distances[i])
    closest_smiles = df['smiles'].iloc[valid_idx[best_idx]]
    actual_gap = df['gap'].iloc[valid_idx[best_idx]]

    print(f"\nCandidate {i+1}:")
    print(f"Target Ideal Gap    : {ideal_gaps_real[i][0]:.4f} Hartrees")
    print(f"Closest Real SMILES : {closest_smiles}")
    print(f"Actual Molecule Gap : {actual_gap:.4f} Hartrees")
    print(f"Validity Status     : 100% (Retrieved via Latent Embedding)")


--- TOP GENERATED CANDIDATES ---

Candidate 1:
Target Ideal Gap    : 0.2022 Hartrees
Closest Real SMILES : CC(CC#N)OCC#C
Actual Molecule Gap : 0.2884 Hartrees
Validity Status     : 100% (Retrieved via Latent Embedding)

Candidate 2:
Target Ideal Gap    : 0.2059 Hartrees
Closest Real SMILES : CC1(O)CC(=O)C11CC1
Actual Molecule Gap : 0.2169 Hartrees
Validity Status     : 100% (Retrieved via Latent Embedding)

Candidate 3:
Target Ideal Gap    : 0.2133 Hartrees
Closest Real SMILES : CC(C)OC(C)C
Actual Molecule Gap : 0.3224 Hartrees
Validity Status     : 100% (Retrieved via Latent Embedding)

Candidate 4:
Target Ideal Gap    : 0.2153 Hartrees
Closest Real SMILES : c1cc(cnc1)C(=O)N
Actual Molecule Gap : 0.2049 Hartrees
Validity Status     : 100% (Retrieved via Latent Embedding)

Candidate 5:
Target Ideal Gap    : 0.2006 Hartrees
Closest Real SMILES : CC1(C)OCCN2CC12
Actual Molecule Gap : 0.3037 Hartrees
Validity Status     : 100% (Retrieved via Latent Embedding)
